### For running QTrack
- Before running this notebook, must have completed era5_regrid.ipynb, regrid_wrfout.ipynb, and wind_combine.ipynb
- Results in one file for each ensemble member: 'wind_for_tracking'+save_name+'.nc'
- All of these wind files have been created already, see directory: /glade/u/home/athornton/qtrack/wind_files/

Note: there are a couple of sections commented out which were used to compare the wrf tracks with reanlysis tracks generated using this tracker. 

In [13]:
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import pandas as pd
import os

from AEW_module import season, AEW, AEW_CCKW

# for regridding
import xesmf as xe
import qtrack
import pickle
from qtrack.curvvort import compute_curvvort
from qtrack.tracking import run_postprocessing, run_tracking

import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter


In [2]:
#### SETTINGS TO CHOOSE ENSEMBLE MEMBER ####

# is this a restart run?
restart = False
# what is the base initialization time? (ens member)
init_time = pd.to_datetime('2020-09-03 12:00:00')
init_string = init_time.strftime('%d%H')
year = init_time.strftime('%Y')
# what time did you turn on the fluxes? Ignore if this is not a restart run...
fluxon_time = pd.to_datetime('2020-09-05 12:00:00')
# this string is used to find that experiment
string = fluxon_time.strftime('%d%H')
# If this is not a restart run, is it control fluxes on or fluxes off?
flux = 'fluxoff' # or 'fluxon'

In [3]:
#### The following code finds the ensemble member based on the settings listed above...
# this is a subdirectory which determines if we are using the long-lived case or the central atlantic case
# initialization date lets the program know which subdirectory to use
if year == '2011':
    subdir = 'long_lived_case'
else:
    subdir = 'cent_atl_case'


# Set directory where wrfout files reside, and list the files for processing.  Set up for a directory with only wrfout files.
if restart == True:
    plt_name = 'rst_on'+string+'z'
    f = pd.Timedelta(init_time - fluxon_time).total_seconds() 
    hours = int((f / 3600)*-1)
    run_name = 'rst_on'+str(hours)
else:
    run_name = flux
    plt_name = run_name
    hours = run_name

os.chdir("/glade/campaign/univ/uncs0067/flux_experiments/"+subdir+"/init"+init_string+"z/"+run_name+"/")
plotsdir = '/glade/u/home/athornton/wrf_visualization/plots/restart/init'+init_string+'z/'+plt_name+'/'

title = init_string+", "+run_name+", "+string+"z"
save_name = run_name +"_"+ init_string
# this is our ensemble member name
save_name

'fluxoff_0312'

In [4]:
path = "/glade/u/home/athornton/qtrack/"
#data_in = path+'wind_files/wind_observations_600_3hr2.nc' # where wind files are located
data_in = path+'wind_files/wind_season_lowres_700_2020_B1-6hr.nc' # where wind files are located

In [5]:
path = "/glade/u/home/athornton/qtrack/"
data_in = path+'wind_files/wind_for_tracking_'+save_name+'.nc' # where wind files are located

In [6]:
curv_file_out = "curv_vort_era5_test.nc"
compute_curvvort(data_in, curv_file_out, njobs_in=-1)

Starting Computation of Radial Averaged CV...
Timestep number: 0
Timestep number: 1
Timestep number: 2
Timestep number: 3
Timestep number: 4
Timestep number: 5
Timestep number: 6
Timestep number: 7
Timestep number: 8
Timestep number: 9
Timestep number: 10
Timestep number: 11
Timestep number: 12
Timestep number: 13
Timestep number: 14
Timestep number: 15
Timestep number: 16
Timestep number: 17
Timestep number: 18
Timestep number: 19
Timestep number: 20
Timestep number: 21
Timestep number: 22
Timestep number: 23
Timestep number: 24
Timestep number: 25
Timestep number: 26
Timestep number: 27
Timestep number: 28
Timestep number: 29
Timestep number: 30
Timestep number: 31
Timestep number: 32
Timestep number: 33
Timestep number: 34
Timestep number: 35
Timestep number: 36
Timestep number: 37
Timestep number: 38
Timestep number: 39
Timestep number: 40
Timestep number: 41
Timestep number: 42
Timestep number: 43
Timestep number: 44
Timestep number: 45
Timestep number: 46
Timestep number: 47
Time

In [7]:
# AEW_raw_save_file = "AEW_tracks_raw_obs.nc"
# run_tracking(input_file=curv_file_out, save_file=AEW_raw_save_file,
#             initiation_bounds=(-170, 40), extrap_longitude_start=-20)


In [8]:
AEW_raw_save_file = "AEW_tracks_raw_"+save_name+".nc"
run_tracking(input_file=curv_file_out, save_file=AEW_raw_save_file,
            initiation_bounds=(-170, 40), extrap_longitude_start=-20, speed_limit_in=False, #threshold_continue=1e-7,
            centroid_radius=646, extrap_dist=2000) ## for 0303_rst_on24 centroid = 646, extrap dist = 800

1 out of 89
2 out of 89
3 out of 89
4 out of 89
5 out of 89
6 out of 89
7 out of 89
8 out of 89
9 out of 89
10 out of 89
11 out of 89
12 out of 89
13 out of 89
14 out of 89
15 out of 89
16 out of 89
17 out of 89
18 out of 89
19 out of 89
20 out of 89
21 out of 89
22 out of 89
23 out of 89
24 out of 89
25 out of 89
26 out of 89
27 out of 89
Something went wrong with centroid, possibly out of bounds
28 out of 89
29 out of 89
30 out of 89
31 out of 89
32 out of 89
33 out of 89
34 out of 89
35 out of 89
36 out of 89
37 out of 89
38 out of 89
39 out of 89
40 out of 89
41 out of 89
42 out of 89
43 out of 89
44 out of 89
45 out of 89
46 out of 89
47 out of 89
48 out of 89
49 out of 89
50 out of 89
51 out of 89
52 out of 89
53 out of 89
54 out of 89
55 out of 89
56 out of 89
57 out of 89
Something went wrong with centroid, possibly out of bounds
Something went wrong with centroid, possibly out of bounds
58 out of 89
Something went wrong with centroid, possibly out of bounds
Something went wron

In [9]:
# AEW_final_nc_file = path+"AEW_tracks_post_proc_obs.nc"
# AEW_final_obj_file = path+"AEW_tracks_post_proc_obs.pkl"
# hov_save_file = path+"final_hovmoller_obs2.png"
# year_in = 2020
# run_postprocessing(input_file=AEW_raw_save_file, real_year_used=year_in, 
#                    curv_data_file=curv_file_out, save_obj_file=AEW_final_obj_file, 
#                    save_nc_file=AEW_final_nc_file, hov_save_file = hov_save_file,
#                    AEW_day_remove=3,
#                   TC_pairing=False, TC_merge_dist=100 )

In [10]:
AEW_final_nc_file = path+"AEW_tracks_post_proc_"+save_name+".nc"
AEW_final_obj_file = path+"AEW_tracks_post_proc_"+save_name+".pkl"
hov_save_file = path+"final_hovmoller_"+save_name+".png"
year_in = year
run_postprocessing(input_file=AEW_raw_save_file, real_year_used=year_in, 
                   curv_data_file=curv_file_out, save_obj_file=AEW_final_obj_file, 
                   save_nc_file=AEW_final_nc_file, hov_save_file = hov_save_file,
                   AEW_day_remove=3)
                  #TC_pairing=True, TC_merge_dist=500, )

Saved


In [11]:
ds = xr.open_dataset(AEW_final_nc_file)

In [12]:
ds

<xarray.Dataset> Size: 151kB
Dimensions:         (system: 14, time: 89, longitude: 139, latitude: 37)
Coordinates:
  * latitude        (latitude) float64 296B 0.0 1.0 2.0 3.0 ... 34.0 35.0 36.0
  * longitude       (longitude) float64 1kB -120.0 -119.0 -118.0 ... 17.0 18.0
  * time            (time) datetime64[ns] 712B 2020-08-24T12:00:00 ... 2020-0...
  * system          (system) float64 112B 1.0 2.0 3.0 4.0 ... 12.0 13.0 14.0
Data variables:
    AEW_lon         (system, time) float64 10kB ...
    AEW_lat         (system, time) float64 10kB ...
    AEW_lon_smooth  (system, time) float64 10kB ...
    AEW_lat_smooth  (system, time) float64 10kB ...
    AEW_strength    (system, time) float64 10kB ...
    TC_gen_time     (system) datetime64[ns] 112B ...
    TC_name         (system) <U3 168B ...
    curv_data_mean  (time, longitude) float64 99kB ...